Исторический исследовательский ноутбук. Сохранён как запись этапов работы; выводы очищены, код не изменён. Ячейки содержат альтернативные сценарии и старые пути. Для повторения актуальных расчётов используйте `dissertation/run_analysis.py` и инструкции в корневом README.


## Влияния зависимости PARL (представленность женщин в парламенте) на всякое

### 0. Библиотеки

In [ ]:
import pandas as pd

from scipy.stats import kruskal

import random

import plotly.graph_objects as go
import plotly.express as px

In [ ]:
import warnings
warnings.filterwarnings("ignore")

### 1. Данные

In [ ]:
file_path = '../data/final_data/final_data.csv'

df = pd.read_csv(file_path, sep = ';', decimal = ',')
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
# кол-во пропусков
print(df.isnull().sum())

### 2. График

In [ ]:
selected_countries = [
    'Nicaragua',
    'United Arab Emirates',
    'Mexico',
    'Moldova'
]

legend_order = (
    df[
        df['Year'].between(2018, 2022) &
        df['Country Name'].isin(selected_countries)
    ]
    .groupby('Country Name')['PARL']
    .mean()
    .sort_values(ascending=False)
)

df_filtered = df[['Country Name', 'Year', 'PARL']].dropna().drop_duplicates()
random_countries = random.sample(legend_order.index.tolist(), k=4)

fig = go.Figure()

for country in legend_order.index:
    country_data = df_filtered[df_filtered['Country Name'] == country]
    fig.add_trace(go.Scatter(
        x=country_data['Year'],
        y=country_data['PARL'],
        mode='lines',
        name=country,
        visible=True if country in random_countries else 'legendonly'
    ))

fig.update_layout(
    template='plotly_white',
    title='Динамика PARL по годам',
    height=750,
    legend_title_text='Страна'
)

fig.show()
fig.write_image("graphs/graphs/PARL.png", width=1600, height=900, scale=4)

### 3. Кластеризация

In [ ]:
# хочу смотреть только последние 5 лет
df = df[df['Year'].between(2018, 2022)]

In [ ]:
variables = [
    'Corruption',
    'Government Eff',
    'Political Stability',
    'Regulatory Quality',
    'Rule of Law',
    'Voice and Accountability'
]

In [ ]:
# Средний PARL по стране за 2018–2022
parl_by_country = df.groupby('Country Name', as_index=False)['PARL'].mean()

# Делим на три группы по перцентилям
low = parl_by_country['PARL'].quantile(0.33)
high = parl_by_country['PARL'].quantile(0.66)

def assign_group(parl):
    if parl <= low:
        return 'Low'
    elif parl <= high:
        return 'Medium'
    else:
        return 'High'

parl_by_country['parl_group'] = parl_by_country['PARL'].apply(assign_group)

# Присоединяем обратно
df = df.merge(parl_by_country[['Country Name', 'parl_group']], on='Country Name', how='left')

### 4. Анализ

In [ ]:
for var in variables:
    fig = px.box(df,
                 x='parl_group', y=var,
                 points='all', notched=True,
                 template='plotly_white',
                 title=f'{var} по группам уровня PARL')

    filename = f"paper1/graphs/PARL_{var}.png"
    fig.write_image(filename, width=1600, height=900, scale=4)
    fig.show()

In [ ]:
order = ['High', 'Medium', 'Low']
df['parl_group'] = pd.Categorical(df['parl_group'], categories=order, ordered=True)

# Сводная таблица: среднее и медиана по группам
summary = df.groupby('parl_group')[variables].agg(['mean', 'median']).round(2)
display(summary)

In [ ]:
# Тест Крускала–Уоллиса по каждой переменной
print("Тест Крускала–Уоллиса:")
for var in variables:
    groups = [group[var].dropna() for name, group in df.groupby('parl_group')]
    stat, p = kruskal(*groups)
    print(f"{var}: H={stat:.2f}, p={p:.4f}")

In [ ]:
# Преобразуем summary в "длинный" формат (только средние)
mean_values = summary.xs('mean', axis=1, level=1).reset_index()
mean_melted = mean_values.melt(id_vars='parl_group', var_name='Indicator', value_name='Mean Value')

fig = px.bar(
    mean_melted,
    x='Indicator',
    y='Mean Value',
    color='parl_group',
    barmode='group',
    title='Средние значения индикаторов по группам уровня PARL',
    height=500
)
fig.update_layout(
    xaxis_title='',
    yaxis_title='Значение индикатора',
    template='plotly_white')

fig.write_image("graphs/graphs/PARL_mean_metrics.png", width=1600, height=900, scale=4)
fig.show()

In [ ]:
# Готовим датафрейм: строки — группы PARL, столбцы — переменные
heatmap_data = summary.xs('mean', axis=1, level=1).copy()
heatmap_data = heatmap_data.loc[['High', 'Medium', 'Low']]  # гарантируем порядок

fig = px.imshow(
    heatmap_data,
    text_auto=True,
    color_continuous_scale='RdBu',
    template='plotly_white',
    aspect='auto',
    title='Средние значения институциональных показателей по группам уровня PARL'
)
fig.update_layout(xaxis_title='Индикаторы', yaxis_title='Группа PARL')

fig.write_image("graphs/graphs/PARL_heatmap_metrics.png", width=1600, height=900, scale=4)
fig.show()

In [ ]:
# Считаем PARL по годам (если ещё не было)
parl_by_year = df.groupby(['Country Name', 'Year'])['PARL'].mean().reset_index()

# Считаем дельту (разницу между макс и мин PARL за годы)
delta_parl = parl_by_year.groupby('Country Name')['PARL'].agg(['min', 'max'])
delta_parl['delta'] = delta_parl['max'] - delta_parl['min']

# Присоединяем группу PARL из уже готового parl_by_country
delta_parl = delta_parl.merge(parl_by_country[['Country Name', 'parl_group']], on='Country Name', how='left')

# Сортировка по дельте
delta_parl_sorted = delta_parl.sort_values(by='delta', ascending=False)

# Выводим
print("Топ стран по росту PARL:")
delta_parl_sorted

In [ ]:
# Moldova
# United Arab Emirates
country = 'United Arab Emirates'
country_data = df[df['Country Name'] == country].sort_values('Year')

# График 1: только PARL
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=country_data['Year'],
    y=country_data['PARL'],
    mode='lines+markers',
    name='PARL (%)',
    line=dict(color='royalblue')
))
fig1.update_layout(
    title=f'{country}: динамика PARL (%)',
    xaxis_title='Год',
    yaxis_title='PARL (%)',
    template='plotly_white',
    xaxis=dict(tickmode='linear', dtick=1),
    height=400
)

# График 2: институциональные показатели
fig2 = go.Figure()
for var in variables:
    fig2.add_trace(go.Scatter(
        x=country_data['Year'],
        y=country_data[var],
        mode='lines+markers',
        name=var
    ))
fig2.update_layout(
    title=f'{country}: институциональные показатели',
    xaxis_title='Год',
    yaxis_title='Индекс',
    template='plotly_white',
    xaxis=dict(tickmode='linear', dtick=1),
    showlegend=True,
    height=400
)

filename1 = f"graphs/graphs/PARL_{country}.png"
filename2 = f"graphs/graphs/PARL_{country}_metrics.png"

fig1.write_image(filename1, width=1600, height=900, scale=4)
fig2.write_image(filename2, width=1600, height=900, scale=4)

fig1.show()
fig2.show()